# 🏛️ AURUM Quantitative Research & Strategy Lab
**Autonomous Dual-Asset Quantitative Intelligence Engine (US100 & XAUUSD)**
---
This notebook provides an institutional research workspace for:
1. **Exploratory Performance Analytics**: Realized R-multiples, Asymmetry Payoff, and Equity Curves.
2. **Monte Carlo Bootstrap Permutations**: Sequence sensitivity and Risk of Ruin.
3. **AI Consensus Layer Testing**: Multi-model trade evaluation (Gemini 2.0 & DeepSeek via OpenRouter).
4. **Key Weekly Liquidity Mapping**: Session High/Low and London Midpoint target models.

In [ ]:
# 1. Imports & Global Environment Setup
import os
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add parent directory to path for src imports
sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

print("✅ Quantitative Research Environment Initialized.")

## 📊 2. Load & Verify Forward Shadow Ledger
Loading verified Gate 2 execution history across US100 and XAUUSD.

In [ ]:
from app import load_shadow_trades

df_trades = load_shadow_trades()
print(f"Total Forward Verified Trades: {len(df_trades)}")
display_cols = ["ticket", "time", "asset", "direction", "entry", "sl", "tp", "pnl", "r", "reason", "balance"]
df_trades[display_cols].tail(10)

## 📈 3. Interactive Dual-Axis Equity & Asymmetry Curve
Visualizing capital growth from the ,000 baseline alongside cumulative R-multiples.

In [ ]:
# Compute Equity Progression
running_bal = 10000.0
running_r = 0.0
eq_points = [{"trade": 0, "balance": 10000.0, "r": 0.0, "label": "Starting Baseline"}]

for idx, row in df_trades.iterrows():
    running_bal += row["pnl"]
    running_r += row["r"]
    eq_points.append({
        "trade": idx + 1,
        "balance": running_bal,
        "r": running_r,
        "label": f"#{row['ticket']} {row['asset']} {row['direction']} ({row['r']:+.2f}R)"
    })

df_eq = pd.DataFrame(eq_points)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(
    x=df_eq["trade"], y=df_eq["balance"],
    mode="lines+markers", name="Portfolio Balance ($)",
    line=dict(color="#10b981", width=3.5),
    marker=dict(size=8, color="#f59e0b"),
    text=df_eq["label"]
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=df_eq["trade"], y=df_eq["r"],
    mode="lines", name="Cumulative R-Multiple",
    line=dict(color="#3b82f6", width=2, dash="dot")
), secondary_y=True)

fig.update_layout(
    title="AURUM Forward Equity Growth & Asymmetric R Profile",
    template="plotly_dark", height=450,
    xaxis_title="Trade Count", yaxis_title="Account Balance ($)"
)
fig.show()

## 🎲 4. Monte Carlo 10,000-Iteration Bootstrap Simulation
Testing sequence permutation sensitivity and estimating probability of drawdown exceeding 5% or 10%.

In [ ]:
from src.monte_carlo import run_monte_carlo

mc_results = run_monte_carlo(df_trades, initial_balance=10000.0, iterations=10000, risk_per_trade=0.0025)
print("=== Monte Carlo Permutation Results ===")
for k, v in mc_results.items():
    print(f"• {k}: {v}")

## 🧠 5. AI Multi-Model Consensus Evaluator
Evaluating live setups using Gemini 2.0 Flash & OpenRouter DeepSeek.

In [ ]:
from src.ai_analyst import MultiModelAIAnalyst

analyst = MultiModelAIAnalyst()
print(f"Gemini Active: {analyst.is_gemini_active()}")
print(f"OpenRouter Active: {analyst.is_openrouter_active()}")

setup_eval = analyst.evaluate_trade_setup(
    asset="US100", direction="SELL",
    entry=31045.51, sl=31069.05, tp=30963.73,
    london_high=31030.0, london_low=30880.0,
    macro_context="Post-London sweep with no Tier-1 news blackout"
)
print("
--- AI Trade Validation Verdict ---")
print(json.dumps(setup_eval, indent=2))